# GRASP: Genetic RAG Attack with Semantic Poisoning
## Implementation with HuggingFace LLaMA-3-8B-Instruct (4-bit Quantized)

This notebook demonstrates the GRASP framework for evaluating RAG system robustness against adversarial poisoning attacks.

**Key Features:**
- LLaMA-3-8B-Instruct with 4-bit quantization
- Genetic Algorithm for adversarial text evolution
- Comprehensive evaluation metrics (ASR, F1, Precision, Recall)
- Batch processing of 100 queries

## Cell 1: Install Dependencies

In [ ]:
# Install required packages
!pip install transformers accelerate bitsandbytes sentence-transformers -q
!pip install beir datasets tqdm -q

print("Dependencies installed successfully!")

## Cell 2: Setup and Imports

In [ ]:
import os
import sys
import json
import torch
import argparse
from pathlib import Path

# Setup paths (adjust for Kaggle input dataset)
# If you uploaded the repository as a Kaggle dataset:
# sys.path.append('/kaggle/input/poisonrag-repo')

# For this demo, assuming code is in working directory
os.chdir('/kaggle/working')

# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Cell 3: Clone Repository (if not using Kaggle dataset)

In [ ]:
# Clone the repository
!git clone https://github.com/iqbal2009048/poisonrag.git
%cd poisonrag

# Add to path
sys.path.insert(0, os.getcwd())

## Cell 4: Prepare Dataset

In [ ]:
from prepare_grasp_data import prepare_100_queries

# Prepare 100 queries from Natural Questions dataset
prepare_100_queries(
    dataset_name='nq',
    split='test',
    seed=42,
    output_dir='data'
)

print("\nDataset preparation complete!")
print("Files created:")
print("  - data/queries_groundtruth.json")
print("  - data/adversarial_samples.json")

## Cell 5: Configure HuggingFace Token (Optional)

In [ ]:
# If you have a HuggingFace token for gated models like LLaMA-3
# Uncomment and add your token:
# HF_TOKEN = "your_huggingface_token_here"

# Update model config
import json

config_path = 'model_configs/llama3_8b_instruct_config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# Update token if needed
# config['api_key_info']['api_keys'][0] = HF_TOKEN

# with open(config_path, 'w') as f:
#     json.dump(config, f, indent=2)

print("Configuration updated!")

## Cell 6: Initialize GRASP Pipeline

In [ ]:
from grasp_kaggle import GRASPPipeline

# Configure pipeline arguments
args = argparse.Namespace(
    dataset='nq',
    split='test',
    num_queries=100,
    num_eval=100,  # Evaluate all 100 queries
    queries_file='data/queries_groundtruth.json',
    adversarial_file='data/adversarial_samples.json',
    model_config_path='model_configs/llama3_8b_instruct_config.json',
    eval_model_code='contriever',
    attack_method='LM_targeted',
    adv_per_query=5,
    top_adversarial=3,  # Use top 3 adversarial texts per query
    use_genetic_algorithm=False,  # Set to True to use GA
    top_k=5,
    score_function='dot',
    orig_beir_results=None,
    seed=42,
    gpu_id=0,
    results_dir='/kaggle/working/results'
)

# Initialize pipeline
pipeline = GRASPPipeline(args)
print("Pipeline initialized!")

## Cell 7: Load Data and Models

In [ ]:
# Phase 1 & 2: Prepare data
pipeline.prepare_data()

# Phase 3: Initialize models (this will take a few minutes)
print("\nLoading models... This may take 5-10 minutes.")
pipeline.initialize_models()

# Phase 4: Load corpus and retrieval results
pipeline.load_corpus_and_results()

print("\nModels and data loaded successfully!")

## Cell 8: Run Clean Baseline Evaluation

In [ ]:
# Phase 5: Run clean RAG baseline
print("Running clean baseline evaluation...")
print("This will generate responses for 100 queries without adversarial injection.")

pipeline.run_clean_baseline()

print("\nClean baseline complete!")
print(f"Results saved to: {args.results_dir}/clean_responses.json")

## Cell 9: Run Poisoned RAG Evaluation

In [ ]:
# Phase 6: Run poisoned RAG evaluation
print("Running poisoned RAG evaluation...")
print("This will inject adversarial texts and evaluate attack effectiveness.")

pipeline.run_poisoned_evaluation()

print("\nPoisoned evaluation complete!")
print(f"Results saved to: {args.results_dir}/poisoned_responses.json")

## Cell 10: Calculate and Display Metrics

In [ ]:
# Phase 7: Evaluate results
pipeline.evaluate_results()

print("\nEvaluation complete!")
print(f"All results saved to: {args.results_dir}/")

## Cell 11: Visualize Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load detailed results
with open(f'{args.results_dir}/evaluation_results.json', 'r') as f:
    eval_results = json.load(f)

# Display summary metrics
print("\n" + "="*60)
print("GRASP EVALUATION SUMMARY")
print("="*60)
print(f"Attack Success Rate (ASR): {eval_results['attack_success_rate']:.2%}")
print(f"Successful Attacks: {eval_results['success_count']}/{eval_results['total_queries']}")
print(f"\nRetrieval Metrics:")
print(f"  Precision: {eval_results['precision_mean']:.4f}")
print(f"  Recall:    {eval_results['recall_mean']:.4f}")
print(f"  F1 Score:  {eval_results['f1_mean']:.4f}")
print("="*60)

# Load CSV for detailed analysis
df = pd.read_csv(f'{args.results_dir}/attack_results_detailed.csv')

# Display sample results
print("\nSample Attack Results:")
print(df[['query_id', 'attack_success', 'num_adv_in_topk']].head(10))

# Plot ASR distribution
plt.figure(figsize=(10, 6))
plt.bar(['Successful', 'Failed'], 
        [eval_results['success_count'], 
         eval_results['total_queries'] - eval_results['success_count']],
        color=['red', 'green'])
plt.title('Attack Success Distribution')
plt.ylabel('Number of Queries')
plt.savefig(f'{args.results_dir}/attack_distribution.png')
plt.show()

print("\nVisualization saved!")

## Cell 12: Examine Sample Attacks

In [ ]:
# Load detailed results
with open(f'{args.results_dir}/detailed_results.json', 'r') as f:
    detailed = json.load(f)

# Show a successful attack example
successful_attacks = [q for q in detailed['per_query_results'] if q['attack_success']]

if successful_attacks:
    print("\nExample Successful Attack:")
    print("="*60)
    example = successful_attacks[0]
    print(f"Query: {example['question']}")
    print(f"\nCorrect Answer: {example['correct_answer']}")
    print(f"Target (Incorrect) Answer: {example['target_answer']}")
    print(f"\nClean Response: {example['clean_response'][:200]}...")
    print(f"\nPoisoned Response: {example['poisoned_response'][:200]}...")
    print(f"\nAdversarial Texts Injected: {example['num_adv_in_topk']}")
    print("="*60)

# Show a failed attack example
failed_attacks = [q for q in detailed['per_query_results'] if not q['attack_success']]

if failed_attacks:
    print("\nExample Failed Attack:")
    print("="*60)
    example = failed_attacks[0]
    print(f"Query: {example['question']}")
    print(f"\nTarget Answer: {example['target_answer']}")
    print(f"\nPoisoned Response: {example['poisoned_response'][:200]}...")
    print(f"\nAdversarial Texts Injected: {example['num_adv_in_topk']}")
    print("="*60)

## Cell 13: Save and Download Results

In [ ]:
# Create a summary report
summary_report = f"""
GRASP Evaluation Report
{'='*60}

Dataset: {args.dataset}
Number of Queries: {eval_results['total_queries']}
Model: LLaMA-3-8B-Instruct (4-bit)
Attack Method: {args.attack_method}
Top-K Retrieved: {args.top_k}
Adversarial Texts per Query: {args.adv_per_query}

Results:
--------
Attack Success Rate: {eval_results['attack_success_rate']:.2%}
Successful Attacks: {eval_results['success_count']}/{eval_results['total_queries']}

Retrieval Metrics:
  Precision: {eval_results['precision_mean']:.4f}
  Recall:    {eval_results['recall_mean']:.4f}
  F1 Score:  {eval_results['f1_mean']:.4f}

{'='*60}
"""

# Save report
with open(f'{args.results_dir}/summary_report.txt', 'w') as f:
    f.write(summary_report)

print(summary_report)
print(f"\nAll results saved to: {args.results_dir}/")
print("\nGenerated files:")
for file in os.listdir(args.results_dir):
    print(f"  - {file}")